# Token-forged MULTI-POST probe (CPU, 0 GPU quota)

Idea A: the single-post replay wastes a wrap-up generation. Does forging the model's analysis channel to COMMIT to posting the marker to N endpoints make gpt_oss keep firing http.post across hops (K posts/candidate) instead of wrapping up after one? Replays at hop-8, counts firing exact-payload posts/candidate. single/prose_multi = controls; forge_* = the novel token-forged inducers.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
def find_repo_root():
    for c in [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/AI-Agent-Security')]:
        if (c/'tools'/'probe_multipost_forge.py').exists(): return c
    raise FileNotFoundError('probe not found; run bootstrap first')
ROOT=find_repo_root(); os.chdir(ROOT); print('repo root:', ROOT)
print('gpu:', 'none (CPU)' if not shutil.which('nvidia-smi') else 'present'); print('cpu:', os.cpu_count())


In [ ]:
os.environ.setdefault('GPT_OSS_GGUF_REPO','unsloth/gpt-oss-20b-GGUF')
os.environ.setdefault('GPT_OSS_GGUF_FILE','gpt-oss-20b-Q4_K_M.gguf')
os.environ.setdefault('LLAMA_CPP_EXTRA_INDEX_URL','https://abetlen.github.io/llama-cpp-python/whl/cpu')


In [ ]:
import importlib.util
def ensure():
    if importlib.util.find_spec('llama_cpp'): print('llama_cpp present'); return
    idx=os.getenv('LLAMA_CPP_EXTRA_INDEX_URL')
    try: subprocess.run([sys.executable,'-m','pip','install','-q','--prefer-binary','llama-cpp-python','--extra-index-url',idx],check=True)
    except subprocess.CalledProcessError:
        e=os.environ.copy(); e['CMAKE_ARGS']='-DGGML_CUDA=off'; e['FORCE_CMAKE']='1'
        subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--force-reinstall','llama-cpp-python'],check=True,env=e)
    assert importlib.util.find_spec('llama_cpp')
ensure()


In [ ]:
import json
cmd=[sys.executable,'tools/probe_multipost_forge.py','--n','10','--urls-per','4','--model','gpt_oss',
     '--budget-per-model','3000','--max-tool-hops','8',
     '--out','research/results/multipost-forge.latest.json','--raw-out','research/results/multipost-forge.raw.jsonl']
print('running:', ' '.join(cmd))
p=subprocess.run(cmd,text=True); print('exit',p.returncode,'(0 = a forged multi-post construction works, 2 = dead)')


In [ ]:
import shutil
s=json.loads(Path('research/results/multipost-forge.latest.json').read_text()); print(json.dumps(s,indent=2,sort_keys=True))
print('\n=== VERDICT: multipost_works =', s['ranking']['multipost_works'])
for r in s['ranking']['ranked']:
    print(f"  {r['construction']:16} mean_posts/cand={r['mean_posts_per_candidate']} max={r['max_posts_per_candidate']} gain_vs_single={r['gain_vs_single']}")
o=Path('/kaggle/working')
if o.exists():
    for f in [Path('research/results/multipost-forge.latest.json'),Path('research/results/multipost-forge.raw.jsonl')]:
        if f.exists() and f.resolve()!=(o/f.name).resolve(): shutil.copy(f,o/f.name)
    print('copied to',o)
